In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# From Next Token machine to Reasoning chatbot

Our pre-trained LLM
- is merely a "Next token prediction" machine

We chart its journey 
- from Next token prediction
- to chatbot 
- to chatbot with tools
- to Chatbot with tools and planning



The journey will continue
- by giving the Chatbot some autonomy
    - replace Human in the loop
    - by a Chatbot with "agency" (to act in place of the human)

<table>
    <center><strong>Next token machine</strong></center>
    <img src="images/next_token_machine.png">
</table>

<table>
    <center><strong>Chat Assistant: No tools</strong></center>
    <img src="images/tools_turns_0.png">
</table>

<table>
    <center><strong>Chatbot Assistant with one tool<br>Assumes fixed location</strong></center>
    <img src="images/tools_turns_1.png">
</table>

<table>
    <center><strong>Chatbot Assistant with one tool<br>Reasons<br>about need to determine user location</strong></center>
    <img src="images/tools_turns_1_.png">
</table>

<table>
    <center><strong>Chatbot Assistant with two tools</strong></center>
    <img src="images/tools_turns_2.png">
</table>

<table>
    <center><strong>Chatbot Assistant with two tools<br>Reasoner<br>Makes a plan</strong></center>
    <img src="images/tools_turns_2_reasoner.png">
</table>

---

# The Anatomy of an AI Agent

## The Biological Analogy

To understand an Agent, we must separate the "Intelligence" from the "Action."

* **The Brain (Assistant Weights):**
    - A "Predict the next token" machine
    - Can only interact with the world by text
    
    - **The Stenographer (Inference Engine):**
        - **Generate Loop**

* **The Hands (Agent Harness):**            
            
   - **Web Browser:** 
       - Allows the agent to access information from the internet
       - interact with web applications
       - perform searches. 

       Extends knowledge beyond the Training cutoff date

    - **Produce artifacts** 
        - Files, documents, slides
        - git commits

    -   **Read/Write Local Files (Code Editor):**
        - **Enables the agent to create, modify, and read files on the local file system.**
            - This is essential for tasks involving data manipulation, code generation, or persistent storage of information.

    -   **Run Command Line Commands (Affect System):** 
        - Provides the agent with the ability to execute system-level commands. 
            - installing packages
            - running scripts
            - managing processes
            - interacting with the operating system directly. 


## The Technical Stack: From Math to Assistant

We build an Agent in three distinct layers:

1. **The Core (The Autoregressive LLM):** 
    - Predict the next token machine
    - Operates in the Generate Loop
2. **The Interface (The Assistant):**
    - Core + Post-training + Personality
        - From "Predict the next token" to "Helpful, conversational Assistant"
        - A "System Prompt" 
            - *"You are a helpful assistant. If you don't know an answer, write a tool call like `get_weather()`."*
3. **The System (The Agentic Harness):**
    - intermediates between the User and the Assistant
        - keeps the Assistant "on track" toward reaching a goal
        - listens for Assistant responses request a tool call
            - pauses the Assistant
            - runs the code
            - injects the response into the conversation (Assistant's context so it sees the result)
       

## The "Secret Sauce": The ReAct Cycle

The Agent 
- is given a mission
- completes it by 
operating according to a **ReAct (Reason + Act)** pattern

ReAct
* **Reason (Thought):** The Brain writes: *"I need to check the temperature in Brooklyn."*
* **Act (Action):** The Brain writes: `get_weather(location="Brooklyn")`.
* **Observe (Observation):** The Hands (Harness) provide the feedback: `15°C`.

The cycle repeats until the **Mission** is accomplished.

In [2]:
import time

def simulate_react_agent(user_input):
    print(f"MISSION STARTED: {user_input}\n")

    # --- TURN 1: REASONING & ACTING ---
    print("--- GENERATE LOOP #1 (Thinking) ---")
    thought = "THOUGHT: I need to check the weather in Brooklyn to answer accurately."
    action  = "ACTION: call_weather_api(location='Brooklyn')"

    for line in [thought, action]:
        for word in line.split():
            print(f"Predicting token: {word.ljust(15)}")
            time.sleep(0.1)
        print("---")

    print("\n[STOP SIGNAL DETECTED: TOOL CALL FOUND]")

    # --- THE GAP: HARNESS EXECUTION ---
    print("\n" + "="*40)
    print("AGENT HARNESS: Contacting OpenWeather API...")
    observation = "65 degrees Farenheit, Light Rain"
    print(f"OBSERVATION: {observation}")
    print("STITCHING FOLLOW-UP PROMPT...")
    print("="*40 + "\n")

    # --- TURN 2: FINALIZING ---
    follow_up = f"{user_input}\n{thought}\n{action}\nObservation: {observation}"
    print("--- GENERATE LOOP #2 (Finalizing) ---")

    final_output = "ANSWER: It is currently 65°F with light rain in Brooklyn. You might want an umbrella!"
    for word in final_output.split():
        print(f"Predicting token: {word.ljust(15)}")
        time.sleep(0.1)

    print("\nMISSION ACCOMPLISHED.")

# Run the simulation
simulate_react_agent("What is the weather in Brooklyn?")

MISSION STARTED: What is the weather in Brooklyn?

--- GENERATE LOOP #1 (Thinking) ---
Predicting token: THOUGHT:       
Predicting token: I              
Predicting token: need           
Predicting token: to             
Predicting token: check          
Predicting token: the            
Predicting token: weather        
Predicting token: in             
Predicting token: Brooklyn       
Predicting token: to             
Predicting token: answer         
Predicting token: accurately.    
---
Predicting token: ACTION:        
Predicting token: call_weather_api(location='Brooklyn')
---

[STOP SIGNAL DETECTED: TOOL CALL FOUND]

AGENT HARNESS: Contacting OpenWeather API...
OBSERVATION: 65 degrees Farenheit, Light Rain
STITCHING FOLLOW-UP PROMPT...

--- GENERATE LOOP #2 (Finalizing) ---
Predicting token: ANSWER:        
Predicting token: It             
Predicting token: is             
Predicting token: currently      
Predicting token: 65°F           
Predicting token: with           
P

# Other Agent Responsibilities

* **Maintain state** 
    - Remember results/artifacts/uploads from previous steps

* **Compress context**
    - Manage context when conversation gets too long for context window

# Code to create diagrams

In [3]:
show = False # do not display; files have already been generated and moved to image directory

In [4]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import os

def draw_no_tools_diagram():
    fig, ax = plt.subplots(figsize=(14, 10))

    # Configuration
    actors = ["USER", "AGENT HARNESS\n(Idle)", "INFERENCE ENGINE\n(The Stenographer)", "ASSISTANT WEIGHTS\n(The Brain)"]
    x_pos = [1, 4.5, 8, 11.5]
    colors = {'user': '#2ecc71', 'harness': '#95a5a6', 'engine': '#9b59b6', 'weights': '#e74c3c'}

    # 1. Draw Actor Headers and Lifelines
    for i, name in enumerate(actors):
        x = x_pos[i]
        ax.plot([x, x], [0, 9.5], color='lightgray', linestyle='--', zorder=0, alpha=0.6)
        color = list(colors.values())[i]
        rect = patches.FancyBboxPatch((x-1.1, 9.7), 2.2, 1.0, boxstyle="round,pad=0.1",
                                     ec="black", fc=color, alpha=0.25)
        ax.add_patch(rect)
        ax.text(x, 10.2, name, ha='center', va='center', weight='bold', fontsize=13)

    # --- STEP 1: INITIAL PROMPT ---
    ax.annotate("1. PROMPT: 'Today's weather is", xy=(8, 9.0), xytext=(1, 9.0),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2.5), fontsize=11, weight='bold')

    # --- PHASE A: REASONING (The Realization) ---
    ax.add_patch(patches.Rectangle((7.5, 7.0), 4.5, 1.5, color='#e74c3c', alpha=0.1))
    ax.text(9.75, 8.0, "PHASE A: REASONING\n\n1. Search for Tools...\n2. [NONE FOUND]\n3. Cannot access real-time data.", 
            color='#c0392b', ha='center', va='center', weight='bold', fontsize=10)

    # --- STEP 2: HARNESS REMAINS IDLE ---
    # We show a "blocked" or "no-op" indicator near the Harness
    ax.text(4.5, 6.0, "Ø NO TOOLS AVAILABLE", ha='center', color='gray', weight='bold', 
            bbox=dict(facecolor='none', edgecolor='gray', linestyle='--'))

    # --- STEP 3: FINAL RESPONSE (The Refusal) ---
    ax.annotate("2. RESPONSE: 'brought to you by Acme tools ...'", 
                xy=(1, 4.0), xytext=(8, 4.0),
                arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=3), 
                fontsize=11, weight='bold')

    # --- FINAL STATE ---
    ax.text(1, 3.0, "CONVERSATION ENDS", color='red', weight='bold', ha='center')

    ax.set_xlim(0, 13); ax.set_ylim(0, 11); ax.axis('off')
    plt.tight_layout()
    
    plt.savefig(os.join("images","next_token_machine.png"))
    plt.show()

if show:
    draw_no_tools_diagram()

In [5]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_no_tools_diagram():
    fig, ax = plt.subplots(figsize=(14, 10))

    # Configuration
    actors = ["USER", "AGENT HARNESS\n(Idle)", "INFERENCE ENGINE\n(The Stenographer)", "ASSISTANT WEIGHTS\n(The Brain)"]
    x_pos = [1, 4.5, 8, 11.5]
    colors = {'user': '#2ecc71', 'harness': '#95a5a6', 'engine': '#9b59b6', 'weights': '#e74c3c'}

    # 1. Draw Actor Headers and Lifelines
    for i, name in enumerate(actors):
        x = x_pos[i]
        ax.plot([x, x], [0, 9.5], color='lightgray', linestyle='--', zorder=0, alpha=0.6)
        color = list(colors.values())[i]
        rect = patches.FancyBboxPatch((x-1.1, 9.7), 2.2, 1.0, boxstyle="round,pad=0.1",
                                     ec="black", fc=color, alpha=0.25)
        ax.add_patch(rect)
        ax.text(x, 10.2, name, ha='center', va='center', weight='bold', fontsize=13)

    # --- STEP 1: INITIAL PROMPT ---
    ax.annotate("1. PROMPT: 'What is the weather?'", xy=(8, 9.0), xytext=(1, 9.0),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2.5), fontsize=11, weight='bold')

    # --- PHASE A: REASONING (The Realization) ---
    ax.add_patch(patches.Rectangle((7.5, 7.0), 4.5, 1.5, color='#e74c3c', alpha=0.1))
    ax.text(9.75, 8.0, "PHASE A: REASONING\n\n1. Search for Tools...\n2. [NONE FOUND]\n3. Cannot access real-time data.", 
            color='#c0392b', ha='center', va='center', weight='bold', fontsize=10)

    # --- STEP 2: HARNESS REMAINS IDLE ---
    # We show a "blocked" or "no-op" indicator near the Harness
    ax.text(4.5, 6.0, "Ø NO TOOLS AVAILABLE", ha='center', color='gray', weight='bold', 
            bbox=dict(facecolor='none', edgecolor='gray', linestyle='--'))

    # --- STEP 3: FINAL RESPONSE (The Refusal) ---
    ax.annotate("2. RESPONSE: 'I am an LLM and cannot\naccess live weather data.'", 
                xy=(1, 4.0), xytext=(8, 4.0),
                arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=3), 
                fontsize=11, weight='bold')

    # --- FINAL STATE ---
    ax.text(1, 3.0, "CONVERSATION ENDS", color='red', weight='bold', ha='center')

    ax.set_xlim(0, 13); ax.set_ylim(0, 11); ax.axis('off')
    plt.tight_layout()
    
    plt.savefig("tools_turns_0.png")
    plt.show()

if show:
    draw_no_tools_diagram()

In [6]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_complete_agent_diagram():
    fig, ax = plt.subplots(figsize=(16, 12))

    # Configuration
    actors = ["USER", "AGENT HARNESS\n(The Hands)", "INFERENCE ENGINE\n(The Stenographer)", "ASSISTANT WEIGHTS\n(The Brain)"]
    x_pos = [1, 4.5, 8, 11.5]
    colors = {'user': '#2ecc71', 'harness': '#3498db', 'engine': '#9b59b6', 'weights': '#e74c3c'}

    # 1. Draw Actor Headers and Lifelines
    for i, name in enumerate(actors):
        x = x_pos[i]
        ax.plot([x, x], [0, 11.5], color='lightgray', linestyle='--', zorder=0, alpha=0.6)
        color = list(colors.values())[i]
        rect = patches.FancyBboxPatch((x-1.1, 11.7), 2.2, 1.0, boxstyle="round,pad=0.1",
                                      ec="black", fc=color, alpha=0.25)
        ax.add_patch(rect)
        ax.text(x, 12.2, name, ha='center', va='center', weight='bold', fontsize=13)

    # PHASE A: REASONING
    ax.annotate("1. PROMPT: 'What is the weather?'", xy=(4.5, 10.5), xytext=(1, 10.5),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2.5), fontsize=12, weight='bold')

    ax.add_patch(patches.Rectangle((7.5, 8.5), 4.5, 1.8, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 10.0, "PHASE A: REASONING (Loop 1)", color='#8e44ad', weight='bold', fontsize=12)

    # STOP SIGNAL
    ax.annotate("4. SIGNAL: <TOOL_CALL>", xy=(4.5, 7.8), xytext=(8, 7.8),
                arrowprops=dict(arrowstyle='-|>', color='red', lw=3))

    # PHASE B: ACTING
    rect_tool = patches.FancyBboxPatch((3.5, 5.5), 2.0, 1.6, boxstyle="round,pad=0.1",
                                       ec="red", fc="#f1c40f", alpha=0.4)
    ax.add_patch(rect_tool)
    ax.text(4.5, 6.3, "PHASE B: ACTING\n(Weather API\ncurrent location)", ha='center', weight='bold', fontsize=12)

    # PHASE C: THE STITCH (FOLLOW-UP PROMPT)
    ax.annotate("6. OBSERVATION -> STITCH\n(Injection)", xy=(8, 4.5), xytext=(4.5, 4.5),
                arrowprops=dict(arrowstyle='fancy', color='blue', lw=2, connectionstyle="arc3,rad=-0.2"),
                fontsize=12, weight='bold', color='blue')

    # PHASE D: FINALIZING
    ax.add_patch(patches.Rectangle((7.5, 2.0), 4.5, 1.8, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 3.5, "PHASE D: FINALIZING (Loop 2)", color='#8e44ad', weight='bold', fontsize=12)

    # FINAL RESPONSE
    ax.annotate("7. FINAL RESPONSE", xy=(1, 1.2), xytext=(8, 1.2),
                arrowprops=dict(arrowstyle='-|>', color='green', lw=3), fontsize=13, weight='bold')

    ax.set_xlim(0, 13); ax.set_ylim(0, 13); ax.axis('off')
    plt.tight_layout()
    
    plt.savefig(os.join("images","tools_turns_1.png"))
    plt.show()

if show:
    draw_complete_agent_diagram()

In [7]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_updated_agent_diagram():
    fig, ax = plt.subplots(figsize=(16, 14))

    # Configuration
    actors = ["USER", "AGENT HARNESS\n(The Hands)", "INFERENCE ENGINE\n(The Stenographer)", "ASSISTANT WEIGHTS\n(The Brain)"]
    x_pos = [1, 4.5, 8, 11.5]
    colors = {'user': '#2ecc71', 'harness': '#3498db', 'engine': '#9b59b6', 'weights': '#e74c3c'}

    # 1. Draw Actor Headers and Lifelines
    for i, name in enumerate(actors):
        x = x_pos[i]
        ax.plot([x, x], [0, 11.5], color='lightgray', linestyle='--', zorder=0, alpha=0.6)
        color = list(colors.values())[i]
        rect = patches.FancyBboxPatch((x-1.1, 11.7), 2.2, 1.0, boxstyle="round,pad=0.1",
                                     ec="black", fc=color, alpha=0.25)
        ax.add_patch(rect)
        ax.text(x, 12.2, name, ha='center', va='center', weight='bold', fontsize=13)

    # --- STEP 1: INITIAL PROMPT ---
    ax.annotate("1. PROMPT: 'What is the weather?'", xy=(8, 11.0), xytext=(1, 11.0),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2.5), fontsize=11, weight='bold')

    # --- PHASE A: INITIAL REASONING ---
    ax.add_patch(patches.Rectangle((7.5, 9.8), 4.5, 1.0, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 10.3, "PHASE A: REASONING\n(Needs Location)", color='#8e44ad', ha='center', weight='bold', fontsize=10)

    # --- STEP 2: ASK USER FOR INFO (Replaces Get_Location Tool) ---
    ax.annotate("2. RESPONSE: 'Please provide your location'", xy=(1, 9.0), xytext=(8, 9.0),
                arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=2), fontsize=10, weight='bold')

    # --- STEP 3: USER PROVIDES DATA ---
    ax.annotate("3. USER INPUT: 'I am in New York'", xy=(8, 8.0), xytext=(1, 8.0),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2), fontsize=10, weight='bold')

    # --- PHASE B: FURTHER REASONING ---
    ax.add_patch(patches.Rectangle((7.5, 6.5), 4.5, 1.0, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 7.0, "PHASE B: REASONING\n(Now has location)", color='#8e44ad', ha='center', weight='bold', fontsize=10)

    # --- STEP 4: TOOL CALL (WEATHER) ---
    ax.annotate("4. SIGNAL: <TOOL_CALL: Get_Weather>", xy=(4.5, 6.0), xytext=(8, 6.0),
                arrowprops=dict(arrowstyle='-|>', color='red', lw=2))

    # --- ACTING (Weather API) ---
    rect_tool2 = patches.FancyBboxPatch((3.5, 5.0), 2.0, 0.8, boxstyle="round,pad=0.1",
                                       ec="red", fc="#f1c40f", alpha=0.4)
    ax.add_patch(rect_tool2)
    ax.text(4.5, 5.4, "ACTING\n(Weather API)", ha='center', weight='bold', fontsize=10)

    # --- STEP 5: STITCH ---
    ax.annotate("5. STITCH (Temp/Conditions)", xy=(8, 4.5), xytext=(4.5, 4.5),
                arrowprops=dict(arrowstyle='fancy', color='blue', lw=1.5, connectionstyle="arc3,rad=-0.1"),
                fontsize=10, weight='bold', color='blue')

    # --- FINAL PHASE ---
    ax.add_patch(patches.Rectangle((7.5, 2.7), 4.5, 1.2, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 3.3, "PHASE C: FINALIZING", color='#8e44ad', ha='center', weight='bold', fontsize=10)

    # FINAL RESPONSE
    ax.annotate("6. FINAL RESPONSE", xy=(1, 1.7), xytext=(8, 1.7),
                arrowprops=dict(arrowstyle='-|>', color='green', lw=3), fontsize=13, weight='bold')

    ax.set_xlim(0, 13); ax.set_ylim(0, 13); ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.join("images","tools_turns_1_.png"))
    
    plt.show()

if show:
    draw_updated_agent_diagram()

In [8]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_complete_agent_diagram():
    fig, ax = plt.subplots(figsize=(16, 14)) # Increased height for more steps

    # Configuration
    actors = ["USER", "AGENT HARNESS\n(The Hands)", "INFERENCE ENGINE\n(The Stenographer)", "ASSISTANT WEIGHTS\n(The Brain)"]
    x_pos = [1, 4.5, 8, 11.5]
    colors = {'user': '#2ecc71', 'harness': '#3498db', 'engine': '#9b59b6', 'weights': '#e74c3c'}

    # 1. Draw Actor Headers and Lifelines
    for i, name in enumerate(actors):
        x = x_pos[i]
        ax.plot([x, x], [0, 11.5], color='lightgray', linestyle='--', zorder=0, alpha=0.6)
        color = list(colors.values())[i]
        rect = patches.FancyBboxPatch((x-1.1, 11.7), 2.2, 1.0, boxstyle="round,pad=0.1",
                                      ec="black", fc=color, alpha=0.25)
        ax.add_patch(rect)
        ax.text(x, 12.2, name, ha='center', va='center', weight='bold', fontsize=13)

    # --- LOOP 1: INITIAL REASONING & TOOL CALL 1 ---
    ax.annotate("1. PROMPT: 'What is the weather?'", xy=(4.5, 11.0), xytext=(1, 11.0),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2.5), fontsize=11, weight='bold')

    ax.add_patch(patches.Rectangle((7.5, 9.8), 4.5, 1.0, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 10.3, "PHASE A: REASONING (Loop 1)", color='#8e44ad', weight='bold', fontsize=10)

    # Tool Call 1
    ax.annotate("2. SIGNAL: <TOOL_CALL: Get_Location>", xy=(4.5, 9.3), xytext=(8, 9.3),
                arrowprops=dict(arrowstyle='-|>', color='red', lw=2))

    # Acting 1
    rect_tool1 = patches.FancyBboxPatch((3.5, 8.3), 2.0, 0.8, boxstyle="round,pad=0.1",
                                       ec="red", fc="#f1c40f", alpha=0.4)
    ax.add_patch(rect_tool1)
    ax.text(4.5, 8.7, "ACTING 1\n(GPS API)", ha='center', weight='bold', fontsize=10)

    # Stitch 1
    ax.annotate("3. STITCH (Location Data)", xy=(8, 7.8), xytext=(4.5, 7.8),
                arrowprops=dict(arrowstyle='fancy', color='blue', lw=1.5, connectionstyle="arc3,rad=-0.1"),
                fontsize=10, weight='bold', color='blue')

    # --- LOOP 2: FURTHER REASONING & TOOL CALL 2 ---
    ax.add_patch(patches.Rectangle((7.5, 6.3), 4.5, 1.0, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 6.8, "PHASE C: REASONING (Loop 2)", color='#8e44ad', weight='bold', fontsize=10)

    # Tool Call 2
    ax.annotate("4. SIGNAL: <TOOL_CALL: Get_Weather>", xy=(4.5, 5.8), xytext=(8, 5.8),
                arrowprops=dict(arrowstyle='-|>', color='red', lw=2))

    # Acting 2
    rect_tool2 = patches.FancyBboxPatch((3.5, 4.8), 2.0, 0.8, boxstyle="round,pad=0.1",
                                       ec="red", fc="#f1c40f", alpha=0.4)
    ax.add_patch(rect_tool2)
    ax.text(4.5, 5.2, "ACTING 2\n(Weather API)", ha='center', weight='bold', fontsize=10)

    # Stitch 2
    ax.annotate("5. STITCH (Temp/Conditions)", xy=(8, 4.3), xytext=(4.5, 4.3),
                arrowprops=dict(arrowstyle='fancy', color='blue', lw=1.5, connectionstyle="arc3,rad=-0.1"),
                fontsize=10, weight='bold', color='blue')

    # --- FINAL PHASE ---
    ax.add_patch(patches.Rectangle((7.5, 2.5), 4.5, 1.2, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 3.1, "PHASE E: FINALIZING", color='#8e44ad', weight='bold', fontsize=10)

    # FINAL RESPONSE
    ax.annotate("6. FINAL RESPONSE", xy=(1, 1.5), xytext=(8, 1.5),
                arrowprops=dict(arrowstyle='-|>', color='green', lw=3), fontsize=13, weight='bold')

    ax.set_xlim(0, 13); ax.set_ylim(0, 13); ax.axis('off')
    plt.tight_layout()
    
    plt.savefig(os.join("images","tools_turns_2.png"))
    plt.show()

if show:
    draw_complete_agent_diagram()

In [9]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_complete_agent_diagram():
    fig, ax = plt.subplots(figsize=(16, 14)) 

    # Configuration
    actors = ["USER", "AGENT HARNESS\n(The Hands)", "INFERENCE ENGINE\n(The Stenographer)", "ASSISTANT WEIGHTS\n(The Brain)"]
    x_pos = [1, 4.5, 8, 11.5]
    colors = {'user': '#2ecc71', 'harness': '#3498db', 'engine': '#9b59b6', 'weights': '#e74c3c'}

    # 1. Draw Actor Headers and Lifelines
    for i, name in enumerate(actors):
        x = x_pos[i]
        ax.plot([x, x], [0, 11.5], color='lightgray', linestyle='--', zorder=0, alpha=0.6)
        color = list(colors.values())[i]
        rect = patches.FancyBboxPatch((x-1.1, 11.7), 2.2, 1.0, boxstyle="round,pad=0.1",
                                     ec="black", fc=color, alpha=0.25)
        ax.add_patch(rect)
        ax.text(x, 12.2, name, ha='center', va='center', weight='bold', fontsize=13)

    # --- LOOP 1: INITIAL COMPREHENSIVE PLANNING ---
    ax.annotate("1. PROMPT: 'What is the weather?'", xy=(4.5, 11.0), xytext=(1, 11.0),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=2.5), fontsize=11, weight='bold')

    # UPDATED: Reasoning Box 1 - Global Plan
    ax.add_patch(patches.Rectangle((7.5, 9.6), 5.2, 1.6, color='#9b59b6', alpha=0.08))
    reasoning_1 = (
        "<think>\n"
        "PLAN:\n"
        "1. I must first determine the location.\n"
        "2. Once I have the location, I will call\n"
        "   a location-aware weather tool.\n"
        "EXECUTION: Calling Get_Location now.\n"
        "</think>"
    )
    ax.text(7.7, 10.4, reasoning_1, color='#8e44ad', weight='bold', fontsize=9, va='center')

    # Tool Call 1
    ax.annotate("2. SIGNAL: <TOOL_CALL: Get_Location>", xy=(4.5, 9.1), xytext=(8, 9.1),
                arrowprops=dict(arrowstyle='-|>', color='red', lw=2))

    # Acting 1
    rect_tool1 = patches.FancyBboxPatch((3.5, 8.1), 2.0, 0.8, boxstyle="round,pad=0.1",
                                       ec="red", fc="#f1c40f", alpha=0.4)
    ax.add_patch(rect_tool1)
    ax.text(4.5, 8.5, "ACTING 1\n(GPS API)", ha='center', weight='bold', fontsize=10)

    # Stitch 1
    ax.annotate("3. STITCH (Location: 'NYC')", xy=(8, 7.6), xytext=(4.5, 7.6),
                arrowprops=dict(arrowstyle='fancy', color='blue', lw=1.5, connectionstyle="arc3,rad=-0.1"),
                fontsize=10, weight='bold', color='blue')

    # --- LOOP 2: EXECUTING THE SECOND STEP OF THE PLAN ---
    # UPDATED: Reasoning Box 2 - Contextual Update
    ax.add_patch(patches.Rectangle((7.5, 6.1), 5.2, 1.4, color='#9b59b6', alpha=0.08))
    reasoning_2 = (
        "<think>\n"
        "Now that I have the location ('NYC'),\n"
        "I can call the location-aware tool\n"
        "to get the weather for this specific area.\n"
        "</think>"
    )
    ax.text(7.7, 6.8, reasoning_2, color='#8e44ad', weight='bold', fontsize=9, va='center')

    # Tool Call 2
    ax.annotate("4. SIGNAL: <TOOL_CALL: Get_Weather('NYC')>", xy=(4.5, 5.6), xytext=(8, 5.6),
                arrowprops=dict(arrowstyle='-|>', color='red', lw=2))

    # Acting 2
    rect_tool2 = patches.FancyBboxPatch((3.5, 4.6), 2.0, 0.8, boxstyle="round,pad=0.1",
                                       ec="red", fc="#f1c40f", alpha=0.4)
    ax.add_patch(rect_tool2)
    ax.text(4.5, 5.0, "ACTING 2\n(Weather API)", ha='center', weight='bold', fontsize=10)

    # Stitch 2
    ax.annotate("5. STITCH (72°F, Sunny)", xy=(8, 4.1), xytext=(4.5, 4.1),
                arrowprops=dict(arrowstyle='fancy', color='blue', lw=1.5, connectionstyle="arc3,rad=-0.1"),
                fontsize=10, weight='bold', color='blue')

    # --- FINAL PHASE ---
    ax.add_patch(patches.Rectangle((7.5, 2.5), 4.5, 1.2, color='#9b59b6', alpha=0.08))
    ax.text(9.75, 3.1, "PHASE E: FINALIZING", color='#8e44ad', weight='bold', fontsize=10, ha='center')

    # FINAL RESPONSE
    ax.annotate("6. FINAL RESPONSE", xy=(1, 1.5), xytext=(8, 1.5),
                arrowprops=dict(arrowstyle='-|>', color='green', lw=3), fontsize=13, weight='bold')

    ax.set_xlim(0, 13); ax.set_ylim(0, 13); ax.axis('off')
    plt.tight_layout()
    
    plt.savefig(os.join("images","tools_turns_2_reasoner.png"))
    
    plt.show()


if show:
    draw_complete_agent_diagram()

In [10]:
print("Done")

Done
